# Optional SQL Warm-Up With Progressive Data Views

This optional warm-up practices SQLite queries against the same learner-facing database path used by the foundations exercises. It is outside the required core module sequence.

In [ ]:
from pathlib import Path
import sqlite3
from tempfile import TemporaryDirectory

import pandas as pd

from banking_fraud_lab import create_minimal_banking_world_sqlite

## Create A Temporary Learner SQLite Database

The SQLite loader excludes protected answer keys by default and creates queryable foundation **Progressive data views**.

In [ ]:
with TemporaryDirectory() as temp_dir:
    connection = create_minimal_banking_world_sqlite(
        Path(temp_dir) / "warmup.sqlite",
        seed=42,
    )
    connection.row_factory = sqlite3.Row
    try:
        table_names = pd.read_sql_query(
            "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name",
            connection,
        )
        view_names = pd.read_sql_query(
            "SELECT name FROM sqlite_master WHERE type = 'view' ORDER BY name",
            connection,
        )
        alert_queue = pd.read_sql_query(
            """
            SELECT
              alert_id,
              institution_name,
              review_priority,
              alert_status,
              suspected_amount_chf
            FROM foundation_alert_lifecycle
            ORDER BY generated_at, alert_id
            """,
            connection,
        )
    finally:
        connection.close()

assert "protected_scenario_answer_keys" not in set(table_names["name"])
assert "foundation_client_relationships" in set(view_names["name"])
assert "foundation_alert_lifecycle" in set(view_names["name"])
assert not alert_queue.empty

alert_queue

The query result is a small alert queue suitable for practicing `SELECT`, ordering, and column selection before the full SQL examples introduce joins, windows, cohorts, and feature extraction.